In [ ]:
# =============================================================
# Lenia Dataset Generator (Entstehung + Kollaps inklusive)
# Output: lenia_train.h5 + lenia_val.h5
# Shape:  (N, T, 64, 64)  float32  in [0, 1]
# =============================================================

import h5py, time, os
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_anim
import IPython.display

# ── Hyperparameter ────────────────────────────────────────────
SIZE         = 64
N_TOTAL      = 2000
T_TRAJ       = 200
T_WARMUP     = 0          # kein Warmup -> Entstehung im Dataset
TRAIN_RATIO  = 0.9
TRAIN_FILE   = 'lenia_train.h5'
VAL_FILE     = 'lenia_val.h5'
N_WORKERS    = os.cpu_count() or 1
MAX_INFLIGHT = N_WORKERS * 2

# Feste Lenia-Parameter
# Update-Regel: A_{t+1} = clip(A_t + (1/T) * G(K*A_t), 0, 1)
# wobei K der Kernel (Nachbarschafts-Gewichtung) und G die Growth-Funktion ist.
R     = 13       # Kernel-Radius in Pixeln. Größer = weiter reichende Nachbarschaft,
                 # langsamere/größere Strukturen. Bei 64x64 Grid sind 10-20 sinnvoll.
T_SIM = 10.0     # Zeitauflösung. dt = 1/T pro Schritt. Größer = feinere Schritte,
                 # stabilere Dynamik, aber langsamere Evolution pro Frame.
M     = 0.135    # Growth-Glocke: Zentrum. Nachbarschafts-Summen nahe M -> Wachstum,
                 # weit weg davon -> Schrumpfen. Steuert die "Wohlfühl-Dichte".
S     = 0.015    # Growth-Glocke: Breite. Klein = strenge Selektion (scharfe Patterns),
                 # groß = tolerantere Dynamik (verwaschene Patterns).
M_K   = 0.5      # Kernel-Glocke: Peak-Position als Bruchteil von R. 0.5 = Ring-Kernel
                 # (Peak bei R/2), 0 wäre Disc-Kernel (Peak im Zentrum).
S_K   = 0.15     # Kernel-Glocke: Breite. Klein = dünner Ring, groß = breiter Ring.

In [ ]:
# ── Lenia-Bausteine ───────────────────────────────────────────
def bell(x, m, s):
    return np.exp(-((x - m) / s) ** 2 / 2)

def make_fK(R, size, m_k=0.5, s_k=0.15):
    mid = size // 2
    D = np.linalg.norm(np.mgrid[-mid:mid, -mid:mid], axis=0) / R
    K = (D < 1) * bell(D, m_k, s_k)
    K /= K.sum()
    return K, np.fft.rfft2(np.fft.fftshift(K))

def lenia_step(A, fK, m, s):
    U = np.fft.irfft2(fK * np.fft.rfft2(A), s=A.shape)
    return np.clip(A + (1.0 / T_SIM) * (bell(U, m, s) * 2 - 1), 0, 1)

K_vis, fK = make_fK(R, SIZE, M_K, S_K)

In [ ]:
# ── Startmatrizen ─────────────────────────────────────────────

def calibrate_density(A, rng, target_range=(0.08, 0.20)):
    # K is normalised, so mean(K*A) ≈ mean(A). Rescaling mean(A) to land
    # near a random target in target_range keeps the expected neighbourhood
    # potential U close to the growth peak M=0.135, ensuring viable dynamics.
    target = rng.uniform(*target_range)
    mean_A = A.mean()
    if mean_A > 1e-6:
        A = np.clip(A * (target / mean_A), 0, 1)
    return A.astype(np.float32)

def sample_blobs_init(rng, size, radius_range=(3, 10), density_range=(0.1, 0.4)):
    A = np.zeros((size, size), dtype=np.float32)
    y, x = np.ogrid[:size, :size]
    n_blobs = rng.integers(2, 6)
    for _ in range(n_blobs):
        cx = rng.integers(8, size - 8)
        cy = rng.integers(8, size - 8)
        radius = rng.uniform(*radius_range)
        density = rng.uniform(*density_range)
        dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
        blob = density * np.exp(-dist**2 / (2 * radius**2))
        A += blob
    return np.clip(A, 0, 1).astype(np.float32)

def sample_noise_init(rng, size, scale_range=(0.02, 0.15)):
    scale = rng.uniform(*scale_range)
    return (rng.random((size, size)) * scale).astype(np.float32)

def sample_sparse_init(rng, size, p=0.03):
    return (rng.random((size, size)) < p).astype(np.float32) * rng.uniform(0.3, 0.8)

def sample_init(rng, size):
    mode = rng.choice(["blobs", "mixed", "noise", "sparse"], p=[0.7, 0.15, 0.1, 0.05])
    if mode == "blobs":
        A = sample_blobs_init(rng, size)
    elif mode == "mixed":
        A = np.clip(
            sample_blobs_init(rng, size) + sample_noise_init(rng, size, (0.01, 0.05)), 0, 1
        ).astype(np.float32)
    elif mode == "noise":
        A = sample_noise_init(rng, size, (0.03, 0.12))
    else:
        A = sample_sparse_init(rng, size)
    return calibrate_density(A, rng)

def is_alive(traj, threshold=1e-3):
    return traj[-1].mean() > threshold and traj[-1].max() > 0.05

def generate_trajectory(seed, max_retries=20):
    for attempt in range(max_retries):
        rng = np.random.default_rng(seed + attempt * 997)
        A = sample_init(rng, SIZE)
        for _ in range(T_WARMUP):
            A = lenia_step(A, fK, M, S)
        traj = np.empty((T_TRAJ, SIZE, SIZE), dtype=np.float32)
        for t in range(T_TRAJ):
            traj[t] = A
            A = lenia_step(A, fK, M, S)
        if is_alive(traj):
            return traj
    return traj  # fallback: return last attempt

In [ ]:
# ── Dataset generieren ───────────────────────────────────────
if __name__ == '__main__':
    print(f'Generiere {N_TOTAL} Trajektorien | T={T_TRAJ} | Warmup={T_WARMUP} | {N_WORKERS} Kerne')
    t0 = time.time()

    data = np.empty((N_TOTAL, T_TRAJ, SIZE, SIZE), dtype=np.float32)
    idx, seed = 0, 0

    pbar = tqdm(total=N_TOTAL)
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        inflight = set()
        for _ in range(min(MAX_INFLIGHT, N_TOTAL)):
            inflight.add(pool.submit(generate_trajectory, seed))
            seed += 1

        while idx < N_TOTAL:
            done, inflight = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                data[idx] = fut.result()
                idx += 1
                pbar.update(1)
                if seed < N_TOTAL:
                    inflight.add(pool.submit(generate_trajectory, seed))
                    seed += 1
    pbar.close()

    dt = time.time() - t0
    print(f'Fertig in {dt:.1f}s  |  {dt/N_TOTAL*1000:.0f} ms/Traj')
    print(f'Shape: {data.shape}  dtype: {data.dtype}  Bereich: [{data.min():.3f}, {data.max():.3f}]')

    # ── Train / Val Split + Speichern ─────────────────────────
    N_train = int(N_TOTAL * TRAIN_RATIO)
    train_data, val_data = data[:N_train], data[N_train:]

    for fname, arr in [(TRAIN_FILE, train_data), (VAL_FILE, val_data)]:
        with h5py.File(fname, 'w') as f:
            f.create_dataset('frames', data=arr,
                             compression='lzf',
                             chunks=(1, T_TRAJ, SIZE, SIZE))
        mb = os.path.getsize(fname) / 1e6
        print(f'Gespeichert: {fname}  ({arr.shape[0]} Trajs, {mb:.1f} MB)')

In [ ]:
# ── Visualisierung: Frame-Raster ──────────────────────────────
VIZ_N, VIZ_T = 6, 8
traj_idx  = np.linspace(0, N_TOTAL - 1, VIZ_N, dtype=int)
frame_idx = np.linspace(0, T_TRAJ - 1, VIZ_T, dtype=int)

vmin, vmax = float(data.min()), float(data.max())

fig, axes = plt.subplots(VIZ_N, VIZ_T, figsize=(VIZ_T * 1.6, VIZ_N * 1.6))
fig.suptitle('Lenia-Trajektorien  (Zeile = 1 Sim, Spalte = Zeit)', fontsize=11)
for row, ti in enumerate(traj_idx):
    for col, fi in enumerate(frame_idx):
        ax = axes[row, col]
        ax.imshow(data[ti, fi], cmap='gray', vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.axis('off')
        if row == 0:
            ax.set_title(f't={fi}', fontsize=8)
plt.tight_layout()
plt.show()

# ── Visualisierung: Eine Trajektorie im Detail ────────────────
SINGLE_IDX  = 0
SINGLE_COLS = 10
SINGLE_ROWS = 4
n_frames = SINGLE_ROWS * SINGLE_COLS
detail_idx = np.linspace(0, T_TRAJ - 1, n_frames, dtype=int)

fig, axes = plt.subplots(SINGLE_ROWS, SINGLE_COLS,
                         figsize=(SINGLE_COLS * 1.3, SINGLE_ROWS * 1.3))
fig.suptitle(f'Trajektorie {SINGLE_IDX} im Detail ({n_frames} Frames ueber T={T_TRAJ})',
             fontsize=11)
for ax, fi in zip(axes.flat, detail_idx):
    ax.imshow(data[SINGLE_IDX, fi], cmap='gray', vmin=vmin, vmax=vmax,
              interpolation='nearest')
    ax.set_title(f't={fi}', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

# ── Visualisierung: Animation ─────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(4, 4))
img = ax2.imshow(data[SINGLE_IDX, 0], cmap='gray', vmin=vmin, vmax=vmax, animated=True)
ax2.set_title(f'Trajektorie {SINGLE_IDX} (animiert)', fontsize=10)
ax2.axis('off')

def anim_update(frame):
    img.set_array(data[SINGLE_IDX, frame])
    return [img]

IPython.display.display(IPython.display.HTML(
    mpl_anim.FuncAnimation(fig2, anim_update, frames=T_TRAJ, interval=20).to_jshtml()
))

# ── Visualisierung: Growth Function & Kernel ──────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
fig.suptitle(f'Lenia-Physik  (R={R}, T={T_SIM}, m={M}, s={S}, m_k={M_K}, s_k={S_K})', fontsize=10)

u = np.linspace(0, 1, 300)
ax1.plot(u, bell(u, M, S) * 2 - 1, 'k', lw=1.5)
ax1.axhline(0, color='gray', lw=0.5, ls='--')
ax1.set_xlim(0, 1)
ax1.set_ylim(-1.1, 1.1)
ax1.set_xlabel('U (Nachbarschaftssumme)')
ax1.set_ylabel('Delta')
ax1.set_title('Growth Function')

ax2.imshow(K_vis, cmap='gray', interpolation='nearest')
ax2.axis('off')
ax2.set_title('Kernel')

plt.tight_layout()
plt.show()